# MoshiRAG retrieval-latency sweep on a Colab T4 (low-memory version)

Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Every step is resumable:
if Colab disconnects, rerun the setup cells (0–5) and then the step you were on — completed runs are skipped.

What runs: MoshiRAG (int8-quantized LM, float16), the same frame-based retrieval-latency injection as the
A100 version, a fixed reference whose embedding is precomputed once. No STT, no retrieval LLM.
See `experiments/async_retrieval_latency/t4/README.md`.

## 0. Check the GPU

In [28]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 1. Settings
Results and the embedding cache go to Google Drive so they survive disconnects.
The Hugging Face cache (~18 GB, or ~30 GB with the encoder) stays on the local disk.

In [29]:
REPO_URL = "https://github.com/zxx3312/moshi-rag.git"   # your fork
BRANCH = "exp/async-retrieval-latency"
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/moshirag_t4"
else:
    WORK = "/content/moshirag_t4"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_home"   # set before anything imports huggingface_hub
REPO = "/content/moshi-rag"
EXP = f"{REPO}/experiments/async_retrieval_latency"
CACHE = f"{WORK}/reference_cache"
RESULTS = f"{WORK}/results"
print(WORK, CACHE, RESULTS, sep="\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/moshirag_t4
/content/drive/MyDrive/moshirag_t4/reference_cache
/content/drive/MyDrive/moshirag_t4/results


## 2. Clone the repository (private fork: use a GitHub token in the URL or `gh auth login`)

In [30]:
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
else:
    # Drop edits made inside Colab (e.g. the hand-applied patch) so the pull cannot conflict.
    # Ignored files (configs/audio link) and everything on Drive are not touched.
    !git -C {REPO} checkout -- . && git -C {REPO} pull --ff-only
!git -C {REPO} log --oneline -1

remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 16 (delta 6), reused 16 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (16/16), 1.03 MiB | 2.45 MiB/s, done.
From https://github.com/zxx3312/moshi-rag
   6a44745..cd2d5f3  exp/async-retrieval-latency -> origin/exp/async-retrieval-latency
Updating 6a44745..cd2d5f3
Fast-forward
 experiments/async_retrieval_latency/.gitignore     |    2 +-
 .../analysis/export_annotation_sheet.py            |   71 +
 .../configs/samples_hard_pilot.jsonl               |   12 +
 .../scripts/build_simpleqa_pilot.py                |   78 +
 experiments/async_retrieval_latency/t4/README.md   |   16 +
 .../async_retrieval_latency/t4/colab_runner.ipynb  |  323 +-
 .../t4/colab_runner_20261005.ipynb                 | 7792 ++++++++++++++++++++
 .../t4/precompute_reference_embeddings.py          |    4 +
 8 files changed, 8253 insertions(+), 45 deletions(-)
 

## 3. Install pinned dependencies (2–4 min; nothing here imports torch in this notebook, so no restart is needed)

In [33]:
!pip install -q -r {EXP}/t4/requirements-colab.txt

In [34]:
!pip uninstall -y -q torchvision
!python -c "import torch, bitsandbytes, xformers, transformers, sphn, huggingface_hub; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0)); print('bnb', bitsandbytes.__version__, '| xformers', xformers.__version__, '| transformers', transformers.__version__, '| hub', huggingface_hub.__version__)"
!python -c "import torch, bitsandbytes as bnb; from bitsandbytes import functional as F; w=torch.randn(256,128,device='cuda',dtype=torch.float16); CB,SCB,_=F.int8_vectorwise_quant(w); s=bnb.MatmulLtState(); s.CB,s.SCB,s.has_fp16_weights=CB,SCB,False; x=torch.randn(4,128,device='cuda',dtype=torch.float16); y=bnb.matmul(x,CB,state=s); r=x.float()@w.float().T; print('int8 matmul on T4 OK, rel err', float((y.float()-r).norm()/r.norm()))"
!python -c "from transformers import AutoTokenizer; print('transformers import OK')"


torch 2.9.1+cu128 | cuda True Tesla T4
bnb 0.49.2 | xformers 0.0.33.post2 | transformers 4.55.3 | hub 0.36.2
int8 matmul on T4 OK, rel err 0.009166198782622814
transformers import OK


## 4. Hugging Face login
Needed only for step 6 (the reference encoder's tokenizer comes from the gated
`meta-llama/Llama-3.2-3B-Instruct`; only its ~9 MB tokenizer files are downloaded, never the Llama weights).
Use a **read** token whose account has accepted the Llama 3.2 license.

In [35]:
from getpass import getpass
from huggingface_hub import login, whoami
login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
print("logged in as", whoami()["name"])


Hugging Face read token: ··········
logged in as zhucynthia3312


## 5. Make the question WAVs (Piper TTS in a separate venv, CPU, ~1 s per question)

In [37]:
import json, pathlib, shutil, subprocess, sys
AUDIO = pathlib.Path(f"{WORK}/audio")   # 存在 Drive 上：只生成一次，--resume 时复用
AUDIO.mkdir(parents=True, exist_ok=True)
link = pathlib.Path(f"{EXP}/configs/audio")
if link.is_dir() and not link.is_symlink():      # 旧版单元格生成的 WAV：保留下来
    for f in link.glob("*.wav"):
        if not (AUDIO / f.name).exists():
            shutil.move(str(f), AUDIO / f.name)
    shutil.rmtree(link)
if not link.is_symlink():
    link.symlink_to(AUDIO, target_is_directory=True)

# Piper 装进单独的文件夹（不用 venv：Colab 的 python 常常缺少 ensurepip）
TTS = "/content/tts_pkgs"
VOICE = "/content/piper_voices/en_US-lessac-medium.onnx"
if not os.path.isdir(f"{TTS}/piper"):
    !pip install -q --target {TTS} piper-tts
env = dict(os.environ, PYTHONPATH=TTS)

def run(cmd):
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return r.stdout

if not os.path.exists(VOICE):
    run([sys.executable, "-m", "piper.download_voices", "en_US-lessac-medium", "--data-dir", "/content/piper_voices"])
for manifest in [f"{EXP}/configs/samples.jsonl", f"{EXP}/t4/configs/samples_small.jsonl"]:
    for line in open(manifest):
        s = json.loads(line)
        wav = (pathlib.Path(manifest).parent / s["wav"]).resolve()
        if not wav.exists():
            print(run([sys.executable, f"{EXP}/scripts/make_tts_question.py", "--voice", VOICE,
                       "--text", s["question"], "--out", str(wav)]).strip())
!ls -la {AUDIO}


total 852
drwx------ 2 root root   4096 Oct  6 00:16 .
drwx------ 5 root root   4096 Oct  6 00:31 ..
-rw------- 1 root root 173468 Oct  6 00:15 capital_australia.wav
-rw------- 1 root root 161208 Oct  6 00:16 longest_river_africa.wav
-rw------- 1 root root 171796 Oct  6 00:16 mars_moons.wav
-rw------- 1 root root 170682 Oct  6 00:16 pride_prejudice_author.wav
-rw------- 1 root root 185728 Oct  6 00:15 tallest_waterfall.wav


In [38]:
!python -c "import numpy, torch; print('numpy', numpy.__version__, numpy.__file__); print('torch', torch.__version__)"


numpy 2.2.6 /usr/local/lib/python3.13/dist-packages/numpy/__init__.py
torch 2.9.1+cu128


## 6. Precompute reference embeddings (one time; separate process, frees the GPU when it exits)
Downloads the MoshiRAG checkpoint (15.4 GB) and the ARC encoder (12.1 GB). On a T4 the encoder autocasts to
float16 (no bf16 support); the official encoder uses bfloat16. For exact parity, compute the cache on an
A100 instead and copy it to `CACHE`.

In [ ]:
p = f"{EXP}/t4/precompute_reference_embeddings.py"
s = open(p).read()
anchor = '    logger.info("ARC weights assigned (%d unexpected keys ignored)", len(result.unexpected_keys))\n'
if "cond.requires_grad_(False)" not in s:
    assert anchor in s, "anchor not found"
    s = s.replace(anchor, anchor + "    cond.requires_grad_(False)\n")
    open(p, "w").write(s)
print("patched" if "cond.requires_grad_(False)" in open(p).read() else "NOT patched")


In [ ]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples.jsonl {EXP}/t4/configs/samples_small.jsonl --cache-dir {CACHE}

## 7. Step 1 — 1 sample × 1 delay (3 s)
Checks: model loads in 16 GB, `[RET]` → injection works, audio + JSONL are written. Read the memory lines in the output.

In [ ]:
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --retrieval-delay 3 --run-name step1_1x1 {ARGS}

In [ ]:
import json
run = f"{RESULTS}/step1_1x1"
print(open(f"{run}/run_status.json").read())
for r in map(json.loads, open(f"{run}/results.jsonl")):
    print({k: r[k] for k in ["run_id", "status", "stop_reason", "rag_triggered", "retrieval_trigger_time",
                             "reference_injection_time", "trigger_to_injection_s", "gpu_peak_allocated_mb", "run_wall_s"]})
    print("pre-RAG :", r["pre_rag_transcript"]); print("post-RAG:", r["post_rag_transcript"])
from IPython.display import Audio
Audio(f"{run}/audio/" + r["run_id"] + "_stereo.wav")

## 8. Step 2 — 1 sample × all delays (0, 1, 2, 3, 5, 8 s)

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/configs/samples.jsonl --run-name step2_1xall {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step2_1xall

## 9. Step 3 — small multi-sample sweep (5 samples × 6 delays = 30 runs)
Long: rerun this cell after a disconnect, it resumes. The 5 questions are examples; choose your own evaluation set.

In [ ]:
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {EXP}/t4/configs/samples_small.jsonl --run-name step3_small {ARGS}

In [ ]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/step3_small

## GPU memory log

In [ ]:
for line in open(f"{RESULTS}/step1_1x1/logs/gpu_memory.jsonl"):
    r = json.loads(line)
    print(f"{r['event']:28s} allocated {r.get('allocated_mb', 0):8.0f} MB  peak {r.get('peak_allocated_mb', 0):8.0f} MB  device {r.get('device_used_mb', 0):8.0f} MB")

In [ ]:
!git -C {REPO} status --short
!git -C {REPO} diff --stat


## 10. Hard-question pilot (12 SimpleQA questions; delays 0 and 8 s first)
Questions the model is unlikely to know, so its behaviour before the reference arrives matters.
Selection: `scripts/build_simpleqa_pilot.py` (rule-based filter + fixed-seed sample, no hand-picking).
References restate question + gold answer only. Run cells 0, 1, 2, 3 (+ check), 4, 5 first.

In [39]:
# WAVs for the pilot questions (stored on Drive through the configs/audio link made in step 5)
import json, os, pathlib, subprocess, sys
TTS = "/content/tts_pkgs"
VOICE = "/content/piper_voices/en_US-lessac-medium.onnx"
if not os.path.isdir(f"{TTS}/piper"):
    !pip install -q --target {TTS} piper-tts
env = dict(os.environ, PYTHONPATH=TTS)
if not os.path.exists(VOICE):
    subprocess.run([sys.executable, "-m", "piper.download_voices", "en_US-lessac-medium", "--data-dir", "/content/piper_voices"], env=env, check=True)
HARD = f"{EXP}/configs/samples_hard_pilot.jsonl"
assert pathlib.Path(f"{EXP}/configs/audio").is_symlink(), "run step 5 first (it links configs/audio to Drive)"
for line in open(HARD):
    s = json.loads(line)
    wav = (pathlib.Path(HARD).parent / s["wav"]).resolve()
    if not wav.exists():
        r = subprocess.run([sys.executable, f"{EXP}/scripts/make_tts_question.py", "--voice", VOICE,
                            "--text", s["question"], "--out", str(wav)], env=env, capture_output=True, text=True)
        print(r.stdout.strip() if r.returncode == 0 else r.stderr)

wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_1009.wav: 4.59 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_1155.wav: 4.39 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_1658.wav: 6.26 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_1838.wav: 7.51 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_2105.wav: 5.17 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_2113.wav: 4.30 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_2885.wav: 4.22 s at 24000 Hz (source 22050 Hz, voice en_US-lessac-medium.onnx)
wrote /content/drive/MyDrive/moshirag_t4/audio/sqa_3564.wav: 5.30 s at 24000 Hz (source 22

In [40]:
!python {EXP}/t4/precompute_reference_embeddings.py --manifest {EXP}/configs/samples_hard_pilot.jsonl --cache-dir {CACHE}

01:49:04 INFO 12 distinct references, 12 to encode, cache: /content/drive/MyDrive/moshirag_t4/reference_cache
01:49:09 INFO NumExpr defaulting to 2 threads.
I0000 00:00:1791251351.447966   37016 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
01:49:17 INFO Loading tokenizer from meta-llama/Llama-3.2-3B-Instruct
01:50:07 INFO ARC weights assigned (2 unexpected keys ignored)
01:50:07 INFO encoder loaded in 60s; GPU allocated 12.17 GB (peak 12.21 GB)
01:50:07 INFO cached ref_52b7f2cd237da452.safetensors: shape (4, 4096) (The Pont Serme is located in the French commune of Coursan.)
01:50:08 INFO cached ref_86aab9884c7c8376.safetensors: shape (3, 4096) (Pedra Branca is the easternmost point of Singapore.)
01:50:08 INFO cached ref_787d54851b56eb55.safetensors: shape (6, 4096) (H

In [45]:
HARD = f"{EXP}/configs/samples_hard_pilot.jsonl"
ARGS = f"--reference-cache {CACHE} --output-root {RESULTS} --resume"
# Extremes first: 12 questions x {0 s, 8 s} = 24 runs (~1 min each). To add the other delays later,
# rerun with "--retrieval-delay 0 1 2 3 5 8": completed runs are skipped.
DELAYS = "0 1 2 3 5 8"
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {HARD} --retrieval-delay {DELAYS} --run-name pilot_hard {ARGS} --dry-run
!python {EXP}/t4/run_latency_sweep_t4.py --manifest {HARD} --retrieval-delay {DELAYS} --run-name pilot_hard {ARGS}

T4 DRY RUN: no torch import, no model loading, no GPU use

[manifest] /content/moshi-rag/experiments/async_retrieval_latency/configs/samples_hard_pilot.jsonl (12/12 samples selected)
  - sqa_1009: 4.588 s audio | reference cached [4, 4096] torch.float32 autocast=float16
  - sqa_1155: 4.391 s audio | reference cached [3, 4096] torch.float32 autocast=float16
  - sqa_1658: 6.26 s audio | reference cached [6, 4096] torch.float32 autocast=float16
  - sqa_1838: 7.514 s audio | reference cached [7, 4096] torch.float32 autocast=float16
  - sqa_2105: 5.169 s audio | reference cached [5, 4096] torch.float32 autocast=float16
  - sqa_2113: 4.298 s audio | reference cached [4, 4096] torch.float32 autocast=float16
  - sqa_2885: 4.217 s audio | reference cached [5, 4096] torch.float32 autocast=float16
  - sqa_3564: 5.296 s audio | reference cached [4, 4096] torch.float32 autocast=float16
  - sqa_3934: 5.029 s audio | reference cached [5, 4096] torch.float32 autocast=float16
  - sqa_4052: 6.028 s audi

In [44]:
!python {EXP}/analysis/analyze_latency.py {RESULTS}/pilot_hard
!python {EXP}/analysis/export_annotation_sheet.py {RESULTS}/pilot_hard
import json
for r in map(json.loads, open(f"{RESULTS}/pilot_hard/results.jsonl")):
    print(f"--- {r['run_id']} | gold: {r['answer'][0]} | [RET] {r['retrieval_trigger_time']} s, injection {r['reference_injection_time']} s")
    print("   pre-RAG :", r["pre_rag_transcript"])
    print("   post-RAG:", r["post_rag_transcript"])

medians in seconds unless noted; rates over ok runs with a defined value
        delay             ok           RET%       ret->inj          onset       1st info  preRAG speech   preRAG words        max gap       ans<inj%      rep.fill%
          0.0             12            1.0           0.48           0.16           2.88            0.4            1.0           0.52            0.0            0.0
          8.0             12            1.0           8.48           0.16              -           6.64           20.0           0.48              -           0.25

prefix consistency (pre-injection tokens identical across delays): 12/12 sample-seeds

wrote /content/drive/MyDrive/moshirag_t4/results/pilot_hard/analysis
wrote /content/drive/MyDrive/moshirag_t4/results/pilot_hard/analysis/annotation_sheet.csv (24 runs)
--- sqa_1009__d8s__s42424242 | gold: Coursan | [RET] 3.68 s, injection 12.16 s
   pre-RAG : Oh, that's a good question. It's in the south of France, nestled in the Pyrenees mount

In [46]:
import os, zipfile, glob
OUT = f"{WORK}/moshirag_t4_results_light.zip"
RUNS = ["step1_1x1", "step2_1xall", "step3_small", "pilot_hard"]
EXAMPLES = ["sqa_3564__d0s__s42424242", "sqa_3564__d8s__s42424242",
            "sqa_2113__d0s__s42424242", "sqa_2113__d8s__s42424242"]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for run in RUNS:
        base = f"{RESULTS}/{run}"
        if not os.path.isdir(base):
            print("missing", run); continue
        for path in glob.glob(f"{base}/**/*", recursive=True):
            if os.path.isdir(path) or path.endswith(".wav") or path.endswith("runner.log"):
                continue
            z.write(path, os.path.relpath(path, RESULTS))
    for ex in EXAMPLES:
        p = f"{RESULTS}/pilot_hard/audio/{ex}_stereo.wav"
        if os.path.exists(p):
            z.write(p, f"pilot_hard/audio_examples/{ex}_stereo.wav")
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), "MB")
from google.colab import files
files.download(OUT)


/content/drive/MyDrive/moshirag_t4/moshirag_t4_results_light.zip 2.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Annotation: download `pilot_hard/analysis/annotation_sheet.csv` from Drive, open it in Google Sheets and fill the label
columns (see `analysis/export_annotation_sheet.py`). More seeds later: rerun the run cell with
`--seeds 42424242 1 2` added; completed runs are skipped.